# Heuers' statistical Football model

The following implementation aims to recreate the Football model of Prof. Dr. Andreas Heuer.

In [19]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from functools import lru_cache
import warnings

NUM_OF_COUNTRIES = 5
NUM_OF_SEASONS = 25
NUM_OF_MATCHES = 9130
NUM_OF_MATCHES_IN_A_SEASON = 1826
SEED = 1893

np.random.seed(SEED)
warnings.filterwarnings('ignore')


In [20]:
# Helperfunctions
def score_from_amount_and_diff(amount, diff):
    if diff < 0:
        home_score = (amount + diff) / 2
        away_score = home_score - diff
    else:
        away_score = (amount - diff) / 2
        home_score = away_score + diff

    # cut excess floating numbers
    if (home_score - int(home_score)) == 0 and (away_score - int(away_score)) == 0:
        home_score = int(home_score)
        away_score = int(away_score)
    
    return home_score, away_score


def print_formatted_pred_score(country, season, matchnumber, ht_name, at_name, ht_pred_score, at_pred_score, ht_actual_score, at_actual_score):
    print("%4s %3s %3s %20s %20s %15s | %6s" %
          (country, season, matchnumber, ht_name, at_name,
           str(ht_pred_score)[0:4] + ":" + str(at_pred_score)[0:4],
           f"{ht_actual_score}:{at_actual_score}"))


def closed_form_solution(X, y):
    # alternative: np.dot(np.linalg.inv(np.dot(X.T, X)), np.dot(X.T, y))
    return np.linalg.solve(np.dot(X.T, X), np.dot(X.T, y))


## Step 1 - Homefieldadvantage (per country and season)

In [21]:
df_homefield = pd.read_csv('data/plot_homefield_adv.csv', sep=';')
avg_homefield_adv = np.sum(df_homefield["homefield_adv"]) / NUM_OF_SEASONS
print("Average homefield advantage on all games:", avg_homefield_adv)


Average homefield advantage on all games: 0.3683735810113519


In [22]:
df_country_season_homefield_adv = pd.read_csv(
    "data/country_season_homefield_adv.csv", sep=";")
df_country_season_homefield_adv['bias'] = 1

noise = np.random.normal(
    avg_homefield_adv, 0.01, df_country_season_homefield_adv.shape[0] * 3)

for index, row in df_country_season_homefield_adv.iterrows():
    if row["hf_adv_actual"] == 0:
        df_country_season_homefield_adv.at[index,
                                           "hf_adv_actual"] = noise[index]
    if row["hf_adv_current_season"] == 0:
        df_country_season_homefield_adv.at[index,
                                           "hf_adv_current_season"] = noise[df_country_season_homefield_adv.shape[0] + index]
    if row["hf_adv_last_three_seasons"] == 0:
        df_country_season_homefield_adv.at[index,
                                           "hf_adv_last_three_seasons"] = noise[df_country_season_homefield_adv.shape[0] * 2 + index]

df_country_season_homefield_adv


,country,season,matchnumber,hf_adv_last_three_seasons,hf_adv_current_season,hf_adv_actual,bias
0,ENG,14,1,0.360328,-0.300000,-0.6,1
1,ENG,14,2,0.369811,0.033333,0.7,1
2,ENG,14,3,0.357811,-0.075000,-0.4,1
3,ENG,14,4,0.357465,0.060000,0.6,1
4,ENG,14,5,0.361030,0.016667,-0.2,1
...,...,...,...,...,...,...,...
925,ITA,18,34,0.317094,0.262857,0.5,1
926,ITA,18,35,0.317094,0.247222,-0.3,1
927,ITA,18,36,0.317094,0.248649,0.3,1
928,ITA,18,37,0.317094,0.268421,1.0,1


In [23]:
#@lru_cache(maxsize=None)
def h_country_season_homefield_adv(country, season, matchnumber):
    if season == 14:
        return avg_homefield_adv
        
    sub_df = df_country_season_homefield_adv.query(
        f"country == '{country}' and season < {season} or (country == '{country}' and season == {season} and matchnumber < {matchnumber})")

    X = sub_df[["hf_adv_last_three_seasons",
                "hf_adv_current_season"]].values # no bias
    y = sub_df["hf_adv_actual"].values

    w = closed_form_solution(X, y)
    # print("hf adv weights:", w)

    input_df = df_country_season_homefield_adv.query(
        f"country == '{country}' and season == {season} and matchnumber == {matchnumber}")
    input = input_df[["hf_adv_last_three_seasons",
                      "hf_adv_current_season"]].values

    return np.dot(w, input.T)[0]

print("Homefield advantage:", h_country_season_homefield_adv("ESP", 18, 2))


Homefield advantage: -0.17982626034923987


## Step 2 - Goalamount per match (per country and season)

there are higher scored countrys as well as lower scoring ones
for now the goalamount will be modeled as a constant

In [24]:
df_goalamount = pd.read_csv('data/plot_goalamount.csv', sep=';')
avg_goalamount = np.sum(df_goalamount["goalamount"]) / NUM_OF_SEASONS
print("Average goalamount of all games:", avg_goalamount)


Average goalamount of all games: 2.724046095631235


In [25]:
df_country_season_goalamount = pd.read_csv(
    "data/country_season_goalamount.csv", sep=";")
df_country_season_goalamount['bias'] = 1

noise = np.random.normal(
    avg_goalamount, 0.01, df_country_season_goalamount.shape[0] * 3)


for index, row in df_country_season_goalamount.iterrows():
    if row["ga_actual"] == 0:
        df_country_season_goalamount.at[index, "ga_actual"] = noise[index]
    if row["ga_current_season"] == 0:
        df_country_season_goalamount.at[index,
                                        "ga_current_season"] = noise[df_country_season_goalamount.shape[0] + index]
    if row["ga_last_three_seasons"] == 0:
        df_country_season_goalamount.at[index,
                                        "ga_last_three_seasons"] = noise[df_country_season_goalamount.shape[0] * 2 + index]

df_country_season_goalamount


,country,season,matchnumber,ga_last_three_seasons,ga_current_season,ga_actual,bias
0,ENG,14,1,2.724707,2.711666,2.6,1
1,ENG,14,2,2.738289,2.600000,2.3,1
2,ENG,14,3,2.712757,2.450000,3.2,1
3,ENG,14,4,2.738602,2.700000,3.0,1
4,ENG,14,5,2.727393,2.775000,3.2,1
...,...,...,...,...,...,...,...
925,ITA,18,34,2.735965,2.663636,2.3,1
926,ITA,18,35,2.735965,2.652941,2.9,1
927,ITA,18,36,2.735965,2.660000,3.1,1
928,ITA,18,37,2.735965,2.672222,2.8,1


In [26]:
#@lru_cache(maxsize=None)
def h_country_season_goalamount(country, season, matchnumber):
    if season == 14:
        return avg_goalamount
        
    sub_df = df_country_season_goalamount.query(
        f"country == '{country}' and season < {season} or (season == {season} and matchnumber < {matchnumber})")

    X = sub_df[["ga_last_three_seasons",
                "ga_current_season"]].values  # no bias
    y = sub_df["ga_actual"].values

    w = closed_form_solution(X, y)
    # print("weights:", w)

    input_df = df_country_season_goalamount.query(
        f"country == '{country}' and season == {season} and matchnumber == {matchnumber}")
    input = input_df[["ga_last_three_seasons",
                    "ga_current_season"]].values

    return np.dot(w, input.T)[0]


print("Goalamount:", h_country_season_goalamount("GER", 16, 1))


Goalamount: 2.936261991159577


## Step 3: Teamstrength and expected goaldifference + Step 4: Specialty for promoters

In [27]:
df_matches = pd.read_csv('data/matches_heuer.csv', sep=';')
df_matches['bias'] = 1

noise = np.random.normal(0, 0.01, df_matches.shape[0] * 6)

for index, row in df_matches.iterrows():
    if row["curr_eff_chancediff"] == 0:
        df_matches.at[index, "curr_eff_chancediff"] = noise[index]
    if row["past_chancediff_combined"] == 0:
        df_matches.at[index,
                           "past_chancediff_combined"] = noise[df_matches.shape[0] + index]
    if row["curr_eff_chanceamount"] == 0:
        df_matches.at[index,
                      "curr_eff_chanceamount"] = noise[df_matches.shape[0] * 2 + index]
    if row["past_chanceamount"] == 0:
        df_matches.at[index,
                           "past_chanceamount"] = noise[df_matches.shape[0] * 3 + index]
    if row["actual_goaldiff"] == 0:
        df_matches.at[index,
                           "actual_goaldiff"] = noise[df_matches.shape[0] * 4 + index]
    if row["actual_goalamount"] == 0:
        df_matches.at[index,
                           "actual_goalamount"] = noise[df_matches.shape[0] * 5 + index]

df_matches


,country,season,matchnumber,teamname,curr_eff_chancediff,past_chancediff_combined,curr_eff_chanceamount,past_chanceamount,market_value_ln_relative,is_promoter,is_home,actual_goalamount,actual_goaldiff,bias
0,ENG,14,1,Queens Park Rangers,-0.005861,-0.006476,0.017177,-0.016672,-0.392415,True,True,1,-1,1
1,ENG,14,1,Hull,-0.001357,-0.016237,-0.001372,0.003611,-0.348740,False,False,1,1,1
2,ENG,14,1,Newcastle,-0.017697,-0.007620,-0.002578,0.010065,-0.169272,False,True,2,-2,1
3,ENG,14,1,Man City,0.004003,-0.012178,0.001265,0.005501,1.002768,False,False,2,2,1
4,ENG,14,1,Burnley,-0.004496,-0.007794,0.004065,-0.004955,-1.140622,True,True,4,-2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18255,ITA,18,38,Napoli,5.242267,5.293421,3.415385,8.877193,1.070083,False,False,5,-1,1
18256,ITA,18,38,Inter,2.881676,2.380263,1.042860,6.754386,1.024225,False,True,3,1,1
18257,ITA,18,38,Empoli,-3.462648,-3.058772,-0.805584,4.403509,-0.738824,True,False,3,-1,1
18258,ITA,18,38,Cagliari,-3.170633,-1.972368,-1.823289,4.824561,-0.358838,False,True,3,-1,1


In [28]:
#@lru_cache(maxsize=None)
def h_teamstrength(country, season, matchnumber, teamname):
    if season == 14 and matchnumber == 1:
        # avg teamstrength
        return 0

    sub_df = df_matches.query(
        f"country == '{country}' and season < {season} and matchnumber == {matchnumber} or (season == {season} and matchnumber < {matchnumber})")

    sub_df["effective_chancediff"] = sub_df["curr_eff_chancediff"]
    X = sub_df[["past_chancediff_combined",
                "curr_eff_chancediff", "market_value_ln_relative"]].values # no bias
    y = []
    # one team is only responsible for half of the goaldiff
    goal_diff = sub_df["actual_goaldiff"].values
    for i in goal_diff:
        y.append(i/2)

    w = closed_form_solution(X, y)
    #print("teamstrength weights:", w)

    input_df = df_matches.query(
        f"season == {season} and matchnumber == {matchnumber} and teamname == '{teamname}'")

    # Homefield advantage correction
    homefield_matches = df_matches.query(
        f"season == {season} and matchnumber < {matchnumber} and teamname == '{teamname}' and is_home").count()["is_home"]

    if matchnumber > 1:
        additional_homefield_matches = homefield_matches - \
            ((matchnumber - 1) - homefield_matches)
        if additional_homefield_matches > 0:
            input_df["curr_eff_chancediff"] -= (additional_homefield_matches * \
                h_country_season_homefield_adv(country, season, matchnumber) * 4) / (matchnumber - 1)

    # Step 4: Specialty for promoters
    if input_df.iloc[0]["is_promoter"]:
        input_df["past_chancediff_combined"] = input_df["past_chancediff_combined"] * 0.45 * w[1]
        input_df["curr_eff_chancediff"] = input_df["curr_eff_chancediff"] * 0.45 * w[1]

    input = input_df[["past_chancediff_combined",
                      "curr_eff_chancediff", "market_value_ln_relative"]].values

    teamstrength = np.dot(w, input.T)[0]

    if input_df.iloc[0]["is_promoter"]:
        teamstrength -= (13/34)

    return teamstrength

#TODO: Current fitness value is missing -> might neglect

print("Teamstrength:", h_teamstrength("ENG", 16, 12, "Leicester"))


Teamstrength: -0.035864796493152086


## Step 5: Expected goalamount per match

In [29]:
#@lru_cache(maxsize=None)
def h_goalamount_from_avg_per_match(country, season, matchnumber, teamname):
    if season == 14 and matchnumber == 1:
        # avg goalamount from avg
        return 0

    sub_df = df_matches.query(
        f"country == '{country}' and season < {season} and matchnumber == {matchnumber} or (season == {season} and matchnumber < {matchnumber})")

    X = sub_df[["past_chanceamount", "curr_eff_chanceamount"]].values  # no bias
    y = []
    for index, row in sub_df.iterrows():
        y.append(row["actual_goalamount"] - avg_goalamount) # - h_country_season_goalamount(row["country"], row["season"], row["matchnumber"]))

    w = closed_form_solution(X, y)
    #print("weights:", w)

    input_df = df_matches.query(
        f"season == {season} and matchnumber == {matchnumber} and teamname == '{teamname}'")
    input = input_df[["past_chanceamount", "curr_eff_chanceamount"]].values

    return np.dot(w, input.T)[0]

print("Goalamount from avg:", h_goalamount_from_avg_per_match("GER", 16, 19, "Bayern"))


Goalamount from avg: 0.21079154127213198


## Step 6: Predicted Goals

In [30]:
def predicted_goals(country, season, matchnumber, hometeam, awayteam):
    goalamount = h_goalamount_from_avg_per_match(
        country, season, matchnumber, hometeam) + h_goalamount_from_avg_per_match(country, season, matchnumber, awayteam) + h_country_season_goalamount(country, season, matchnumber)
    #goaldiff = h_country_season_homefield_adv(country, season, matchnumber) + h_teamstrength(country, season, matchnumber, hometeam) - h_teamstrength(country, season, matchnumber, awayteam)
    goaldiff = avg_homefield_adv + \
        h_teamstrength(country, season, matchnumber, hometeam) - \
        h_teamstrength(country, season, matchnumber, awayteam)
    return (goalamount, goaldiff)


## Step 7: Dependence on matchnumber

In [31]:
def matchnumber_goalamount_multiplier(country, matchnumber):
    # Uncertain
    if country == 'GER':
        if matchnumber < 18:
            return 1.02
        elif matchnumber < 31:
            return 0.95
        else:
            return 1.2
    else:
        if matchnumber < 19:
            return 1.02
        elif matchnumber < 35:
            return 0.95
        else:
            return 1.2


## Final: Full run on dataset to measure goalamount and goaldifference error

In [32]:
def predict(country, season, matchnumber, hometeam, awayteam):
    amount, diff = predicted_goals(country, season, matchnumber, hometeam, awayteam)

    home_score, away_score = score_from_amount_and_diff(amount, diff)

    home_score *= matchnumber_goalamount_multiplier(country, matchnumber)
    away_score *= matchnumber_goalamount_multiplier(country, matchnumber)

    return (home_score+ away_score), (home_score - away_score)


In [33]:
# For match in Dataset run prediction and measure error
do_a_whole_run = True

if not do_a_whole_run:
    print("Skipped whole run")

test_df = df_matches.query(f"season == 18")
ga_squared_error, gd_squared_error = 0, 0

if do_a_whole_run:
    for index, row in test_df.iterrows():
        if index % 2 == 0:
            home_score, away_score = score_from_amount_and_diff(
                row["actual_goalamount"], row["actual_goaldiff"])
            hometeam = row["teamname"]
            continue
        else:
            awayteam = row["teamname"]
            ga_pred, gd_pred = predict(
                row["country"], row["season"], row["matchnumber"], hometeam, awayteam)
            ga_squared_error += np.square(ga_pred - row["actual_goalamount"])
            gd_squared_error += np.square(gd_pred - row["actual_goaldiff"])
            h_score_pred, a_score_pred = score_from_amount_and_diff(ga_pred,gd_pred)
            print_formatted_pred_score(row["country"], row["season"], row["matchnumber"], hometeam, awayteam, h_score_pred ,a_score_pred, home_score, away_score)

    print("Squared error goalamount",
          ga_squared_error / NUM_OF_MATCHES_IN_A_SEASON)
    print("Squared error goaldiff", gd_squared_error / NUM_OF_MATCHES_IN_A_SEASON)


 ENG  18   1               Wolves              Everton       1.42:1.63 |    2:2
 ENG  18   1              Watford             Brighton       1.99:1.46 |    2:0
 ENG  18   1          Southampton              Burnley       1.87:1.35 |    0:0
 ENG  18   1            Newcastle            Tottenham       1.61:1.76 |    1:2
 ENG  18   1              Man Utd            Leicester       2.05:1.39 |    2:1
 ENG  18   1              Arsenal             Man City       1.93:1.82 |    0:2
 ENG  18   1         Huddersfield              Chelsea       1.51:2.01 |    0:3
 ENG  18   1               Fulham       Crystal Palace       1.52:1.59 |    0:2
 ENG  18   1          Bournemouth              Cardiff       2.02:0.84 |    2:0
 ENG  18   1            Liverpool             West Ham       2.29:1.38 |    4:0
 ENG  18   2            Tottenham               Fulham       1.97:0.47 |    3:1
 ENG  18   2              Everton          Southampton       1.61:0.87 |    2:1
 ENG  18   2             West Ham       